In [ ]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="gpt-4o-mini")

system_prompt = (
  "You're a helpful AI assistant. Given a user question "
  "and some Wikipedia article snippets, answer the user "
  "question. If none of the articles answer the question, "
  "just say you don't know."
  "\n\nHere are the Wikipedia articles: "
  "{context}"
)

retriever = WikipediaRetriever(top_k_results=6, doc_content_chars_max=2000)
prompt = ChatPromptTemplate.from_messages(
  [
    ("system", system_prompt),
    ("human", "{input}"),
  ]
)
                                                                         

In [ ]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="gpt-4o-mini")

system_prompt = (
  "You're a helpful AI assistant. Given a user question "
  "and some Wikipedia article snippets, answer the user "
  "question. If none of the articles answer the question, "
  "just say you don't know."
  "\n\nHere are the Wikipedia articles: "
  "{context}"
)

retriever = WikipediaRetriever(top_k_results=6, doc_content_chars_max=2000)
prompt = ChatPromptTemplate.from_messages(
  [
    ("system", system_prompt),
    ("human", "{input}"),
  ]
)

In [ ]:
result = chain.invoke({"input": "How did the USA fair at the 2024 
  Summer Olympics"})
print(result.keys())

In [ ]:
from typing import List
from langchain_core.pydantic_v1 import BaseModel, Field


class CitedAnswer(BaseModel):
  """Answer the user question based only on the given sources, and cite 
    the sources used."""

  answer: str = Field(
    ...,
    description="The answer to the user question, which is based only on 
      the given sources.",
  )
  citations: List[int] = Field(
    ...,
    description="The integer IDs of the SPECIFIC sources which justify 
      the answer.",
  )

In [ ]:
structured_llm = llm.with_structured_output(CitedAnswer)

query = """How did the USA fair at the 2024 Summer Olympics"""
result = structured_llm.invoke(query)

resultclass Citation(BaseModel):
  source_id: int = Field(
    ...,
    description="The integer ID of a SPECIFIC source which 
      justifies the answer.",
  )
  quote: str = Field(
    ...,
    description="The VERBATIM quote from the specified source that 
      justifies the answer.",
  )


class QuotedAnswer(BaseModel):
  """Answer the user question based only on the given sources, and 
    cite the sources used."""

  answer: str = Field(
    ...,
    description="The answer to the user question, which is based 
      only on the given sources.",
  )
  citations: List[Citation] = Field(
    ..., description="Citations from the given sources that 
      justify the answer."
  )


In [ ]:
rag_chain = (
  RunnablePassthrough.assign(context=(lambda x: 
    format_docs_with_id(x["context"])))
  | prompt
  | llm.with_structured_output(QuotedAnswer)
)

retrieve_docs = (lambda x: x["input"]) | retriever

chain = RunnablePassthrough.assign(context=retrieve_docs).assign(
  answer=rag_chain
)

chain.invoke({"input": "How did the USA fair at the 2024 Summer
  Olympics"})
